# Monthly Listening Clustering

This notebook uses K-Means to group months with similar listening behavior.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

processed = BASE_DIR / 'data' / 'processed'
figures = BASE_DIR / 'reports' / 'figures'
figures.mkdir(parents=True, exist_ok=True)


In [ ]:
monthly = pd.read_csv(processed / 'spotify_monthly_behavior_features.csv')
monthly = monthly.sort_values('month').reset_index(drop=True)

print(monthly.shape)
print(monthly.head())


## 1. Select features

Each row is one month. The features describe listening behavior during that month.

In [ ]:
features = [
    'total_plays', 'unique_tracks', 'unique_artists',
    'total_minutes', 'average_minutes', 'skip_rate',
    'night_listening_ratio', 'weekend_listening_ratio',
    'repeat_rate', 'discovery_rate'
]

X = monthly[features].copy()
print(X.head())


## 2. Scale the features

K-Means uses distance, so the features are scaled to a similar range.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled data shape:', X_scaled.shape)


## 3. Find a suitable number of clusters

We compare K values using the silhouette score.

In [ ]:
max_k = min(8, len(monthly) - 1)
scores = {}

for k in range(2, max_k + 1):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

print('Silhouette scores:')
for k, score in scores.items():
    print(k, ':', round(score, 3))


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(list(scores.keys()), list(scores.values()), marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Silhouette score')
plt.title('Choosing K with Silhouette Score')
plt.xticks(list(scores.keys()))
plt.tight_layout()
plt.savefig(figures / 'silhouette_scores.png')
plt.show()


## 4. Apply K-Means

We use the K with the highest silhouette score as a starting point.

In [ ]:
best_k = max(scores, key=scores.get)
print('Selected K:', best_k)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
monthly['cluster'] = kmeans.fit_predict(X_scaled)

print(monthly[['month', 'cluster']].head(15).to_string(index=False))


## 5. Look at the cluster profiles

The mean of each feature helps us understand how the clusters differ.

In [ ]:
cluster_profile = monthly.groupby('cluster')[features].mean().round(3)
print(cluster_profile)


## 6. Cluster timeline

This shows which cluster each month belongs to.

In [ ]:
plt.figure(figsize=(14, 4))
plt.scatter(range(len(monthly)), monthly['cluster'])
plt.yticks(sorted(monthly['cluster'].unique()))
plt.xlabel('Month index')
plt.ylabel('Cluster')
plt.title('Monthly Listening Behavior Clusters')
plt.tight_layout()
plt.savefig(figures / 'monthly_clusters.png')
plt.show()


## 7. Save the result

The cluster column will be used later by the project and Streamlit app.

In [ ]:
output = processed / 'spotify_monthly_clusters.csv'
monthly.to_csv(output, index=False)
print('Saved:', output)


## Final observations

After running the notebook, describe the clusters using the actual numbers:

- Selected K: ______
- Best silhouette score: ______
- Cluster 0 pattern: ______
- Cluster 1 pattern: ______
- Other cluster patterns if present: ______
- What changed over time? ______
